In [23]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from typing import TypedDict
from dotenv import load_dotenv
import os

In [24]:
load_dotenv()
print(os.getenv("OPENAI_API_KEY") is not None)

True


## Define State

In [25]:
class LLMState(TypedDict):
    question: str
    answer: str

## Define LLM

In [26]:
model = ChatOpenAI(model_name="gpt-5-nano",  
 temperature=0.7, max_tokens=1000)

## Creating Node

In [27]:
def llm_qa(state: LLMState) -> LLMState:
    question = state["question"]
    answer = model.invoke(f"Answer the following question: {question}")
    return {"question": question, "answer": answer}

Quick questions
 1) model.predict() vs model.invoke()?


## Building Graph

In [28]:
graph = StateGraph(LLMState)

graph.add_node("llm_qa",llm_qa)

graph.add_edge(START, "llm_qa")
graph.add_edge("llm_qa", END)

workflow = graph.compile()

## Execute workflow

In [31]:
initial_state = {"question": "Who is the president minister of india?", }
final_state = workflow.invoke(initial_state)
print(final_state)

{'question': 'Who is the president minister of india?', 'answer': AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 1000, 'prompt_tokens': 19, 'total_tokens': 1019, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 1000, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-Dwemah48VPrgPGUfHIqehUWzh3zqr', 'service_tier': 'default', 'finish_reason': 'length', 'logprobs': None}, id='lc_run--019f1b76-44f8-7180-b394-3bb1014ae42d-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 19, 'output_tokens': 1000, 'total_tokens': 1019, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 1000}})}
